# Problem 11 (100 points)

Physics-Informed Neural Networks (PINNs) use deep learning to solve partial differential equations (PDEs). Instead of discretizing the domain (as in finite differences or finite elements), a PINN trains a neural network $U(t, x \mid \theta)$ whose loss function enforces the PDE, initial conditions (IC), and boundary conditions (BC). The key computational tool is `torch.autograd.grad` with `create_graph=True`, which enables computing higher-order derivatives of the network output with respect to its inputs.

In this problem, you will build a PINN to solve the **1D heat equation** on a rod of unit length:

$$u_t = \alpha \, u_{xx}, \quad x \in [0, 1], \quad t \in [0, 1]$$

with IC $u(0, x) = \sin(\pi x)$ and BC $u(t, 0) = u(t, 1) = 0$.

We use the following notation in this problem.
- $u(t, x)$ — temperature at time $t$ and position $x$.
- $u_t = \frac{\partial u}{\partial t}$, $u_x = \frac{\partial u}{\partial x}$, $u_{xx} = \frac{\partial^2 u}{\partial x^2}$.
- $\alpha > 0$ — thermal diffusivity.
- $U(t, x \mid \theta)$ — neural network approximation of $u$.
- $\mathcal{D}_{\text{PDE}}$, $\mathcal{D}_{\text{IC}}$, $\mathcal{D}_{\text{BC}}$ — collocation point sets for the PDE, IC, and BC.

In [ ]:
# Run code in this cell

"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""

import torch
import torch.nn as nn
import torch.optim as optim
import torch.autograd as autograd
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

> WARNING !!!
>
- Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
    - **As a part of your final solution.**
    - **Temporarily import something to assist you to get a solution.**

## Part 1 (10 points, non-coding task)

**Do the following tasks (Reasoning is required).**

Verify that the closed-form solution to the heat equation with the given IC and BC is:

$$u(t, x) = e^{-\alpha \pi^2 t} \sin(\pi x)$$

1. Compute $u_t$ and $u_{xx}$ from this formula.
2. Show that $u_t - \alpha u_{xx} = 0$.
3. Verify $u(0, x) = \sin(\pi x)$.
4. Verify $u(t, 0) = u(t, 1) = 0$.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

Before building the full PINN, let us learn how to compute derivatives with `torch.autograd.grad`.

## Part 2 (10 points, coding task)

**Do the following tasks.**

Implement `compute_derivatives(u, x)` that computes $\frac{\partial u}{\partial x}$ using `torch.autograd.grad`.

- `u`: tensor of shape `(N, 1)` — network output.
- `x`: tensor of shape `(N, 1)` with `requires_grad=True` — input.
- Returns: tensor of shape `(N, 1)` — the derivative $\frac{\partial u}{\partial x}$.

Key requirement: use `create_graph=True` so that higher-order derivatives can be computed by calling this function on the output again.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def compute_derivatives(u, x):
    """
    Compute du/dx using autograd.
    u: (N, 1), x: (N, 1) with requires_grad=True
    Returns: (N, 1)
    """
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
# Test: u = x^2 -> du/dx = 2x
x_test = torch.linspace(-2, 2, 100).reshape(-1, 1).requires_grad_(True)
u_test = x_test ** 2
du_dx = compute_derivatives(u_test, x_test)
assert torch.allclose(du_dx, 2 * x_test, atol=1e-5), f"Max error: {(du_dx - 2*x_test).abs().max():.6f}"

# Test higher-order: d2u/dx2 = 2
d2u_dx2 = compute_derivatives(du_dx, x_test)
assert torch.allclose(d2u_dx2, 2 * torch.ones_like(x_test), atol=1e-4), "Second derivative should be 2"

print("Part 2 passed!")

Now let us build the neural network architecture for the PINN.

## Part 3 (10 points, coding task)

**Do the following tasks.**

Build a `HeatPINN` network.

- Input: concatenation of $t$ and $x$ → shape `(N, 2)`.
- Architecture: `Linear(2, 64) -> Tanh -> Linear(64, 64) -> Tanh -> Linear(64, 1)`.
- Output: predicted temperature $U(t, x)$ → shape `(N, 1)`.

Note: Tanh is preferred over ReLU because PINNs require smooth (twice-differentiable) activation functions. $\text{ReLU}''(x) = 0$ almost everywhere, making it unsuitable for computing $u_{xx}$.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

class HeatPINN(nn.Module):
    def __init__(self):
        super().__init__()
        ...
    
    def forward(self, tx):
        """tx: (N, 2) where tx[:, 0]=t, tx[:, 1]=x -> (N, 1)"""
        ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
model = HeatPINN()
tx = torch.rand(50, 2, requires_grad=True)
u = model(tx)
assert u.shape == (50, 1), f"Expected (50, 1), got {u.shape}"

# Gradients should flow
u.sum().backward()
assert tx.grad is not None
print("Part 3 passed!")

The PDE loss enforces $u_t - \alpha u_{xx} = 0$ at interior collocation points.

## Part 4 (15 points, coding task)

**Do the following tasks.**

Implement `compute_pde_residual(model, tx, alpha)` that:

1. Passes `tx` (shape `(N, 2)`, `requires_grad=True`) through the model to get $U$.
2. Uses `compute_derivatives` to compute $U_t$ (derivative w.r.t. $t$, i.e., column 0 of `tx`).
3. Uses `compute_derivatives` to compute $U_x$ (derivative w.r.t. $x$, i.e., column 1 of `tx`).
4. Uses `compute_derivatives` again on $U_x$ to get $U_{xx}$.
5. Returns the residual $r = U_t - \alpha U_{xx}$ of shape `(N, 1)`.

Hint: to get the derivative w.r.t. a specific column, you need `autograd.grad(u.sum(), tx, create_graph=True)[0]` and then index the appropriate column.

In [ ]:
### WRITE YOUR SOLUTION HERE ###

def compute_pde_residual(model, tx, alpha=0.1):
    """
    Compute PDE residual: u_t - alpha * u_xx
    tx: (N, 2) with requires_grad=True
    Returns: (N, 1) residual
    """
    ...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
torch.manual_seed(42)
model = HeatPINN()
tx = torch.rand(100, 2, requires_grad=True)
residual = compute_pde_residual(model, tx, alpha=0.1)
assert residual.shape == (100, 1), f"Expected (100, 1), got {residual.shape}"

# Residual should be differentiable (for backprop)
loss = (residual ** 2).mean()
loss.backward()
has_grad = any(p.grad is not None and p.grad.abs().sum() > 0 for p in model.parameters())
assert has_grad, "Gradients should flow to model parameters"
print(f"Part 4 passed! PDE residual mean: {residual.abs().mean().item():.6f}")

The total PINN loss combines three terms: PDE residual, IC error, and BC error.

## Part 5 (20 points, coding task)

**Do the following tasks.**

Implement the full PINN training loop.

1. Create collocation points:
   - $\mathcal{D}_{\text{PDE}}$: 500 random points in $[0,1]^2$, with `requires_grad=True`.
   - $\mathcal{D}_{\text{IC}}$: 101 points at $t = 0$, $x \in \{0, 0.01, \ldots, 1\}$.
   - $\mathcal{D}_{\text{BC}}$: 202 points at $x \in \{0, 1\}$, $t \in \{0, 0.01, \ldots, 1\}$.

2. Losses:
   - $L_{\text{PDE}} = \frac{1}{N}\sum r_i^2$ (mean squared residual over $\mathcal{D}_{\text{PDE}}$).
   - $L_{\text{IC}} = \frac{1}{N}\sum (U(0, x) - \sin(\pi x))^2$ over $\mathcal{D}_{\text{IC}}$.
   - $L_{\text{BC}} = \frac{1}{N}\sum U(t, x_{\text{boundary}})^2$ over $\mathcal{D}_{\text{BC}}$.
   - $L_{\text{total}} = L_{\text{PDE}} + L_{\text{IC}} + L_{\text{BC}}$.

3. Train for 2000 epochs with Adam at `lr=1e-3`, $\alpha = 0.1$.
4. Store loss history as `loss_history` (list of floats).

In [ ]:
### WRITE YOUR SOLUTION HERE ###

torch.manual_seed(42)
alpha = 0.1
pinn_model = HeatPINN()

# Create collocation points
...

# Training loop
loss_history = []
...

""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
assert len(loss_history) == 2000
assert loss_history[-1] < loss_history[0], "Loss should decrease"
assert loss_history[-1] < 0.01, f"Final loss {loss_history[-1]:.6f} should be < 0.01"

# Check IC satisfaction
pinn_model.eval()
x_ic = torch.linspace(0, 1, 50).reshape(-1, 1)
t_ic = torch.zeros(50, 1)
tx_ic = torch.cat([t_ic, x_ic], dim=1)
with torch.no_grad():
    u_ic = pinn_model(tx_ic)
expected_ic = torch.sin(np.pi * x_ic)
ic_error = (u_ic - expected_ic).abs().mean().item()
print(f"Part 5 passed! Final loss: {loss_history[-1]:.6f}, IC error: {ic_error:.4f}")

Let us evaluate the trained PINN against the known analytical solution.

## Part 6 (15 points, coding task)

**Do the following tasks.**

1. Generate a test grid: $t, x \in \{0, 0.01, \ldots, 1\}$ (10201 points total). Store as `tx_test` of shape `(10201, 2)`.
2. Compute the analytical solution $u_{\text{exact}}(t, x) = e^{-\alpha \pi^2 t} \sin(\pi x)$. Store as `u_exact` of shape `(10201,)`.
3. Compute the PINN prediction $U(t, x)$ on the test grid (in eval mode, no grad). Store as `u_pred` of shape `(10201,)`.
4. Compute and print the MSE between `u_exact` and `u_pred`.
5. Create two side-by-side scatter plots of $(t, x)$ colored by temperature:
   - Left: ground truth $u_{\text{exact}}$.
   - Right: PINN prediction $U$.
   - Use `cmap='viridis'` and `plt.colorbar(label='Temperature')`.

In [ ]:
### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

In [ ]:
""" VERIFICATION """
assert tx_test.shape == (10201, 2)
assert u_exact.shape == (10201,)
assert u_pred.shape == (10201,)
mse = ((u_exact - u_pred) ** 2).mean().item()
assert mse < 0.01, f"MSE {mse:.6f} should be < 0.01"
print(f"Part 6 passed! MSE: {mse:.6f}")

## Part 7 (10 points, non-coding task)

**Do the following tasks (Reasoning is required).**

1. Why must we use `create_graph=True` in `torch.autograd.grad` for PINNs? What fails if we set it to `False`?

2. PINNs use Tanh activation instead of ReLU. What is the mathematical reason? (Hint: consider $\text{ReLU}''(x)$.)

3. The PINN loss has three terms: $L_{\text{PDE}}$, $L_{\text{IC}}$, $L_{\text{BC}}$. In the official USAAIO problem, IC and BC are given higher weight. Why is it common to weight IC/BC more heavily than the PDE residual?

4. We use all IC/BC data points in every training step rather than mini-batching them. Why is this important for PINNs?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 8 (10 points, non-coding task)

**Do the following tasks (Reasoning is required).**

Consider extending the PINN to the **2D wave equation**: $u_{tt} = c^2(u_{xx} + u_{yy})$ on domain $[0,1]^2 \times [0, T]$.

1. What is the input dimension of the network? What are the inputs?
2. List all the derivatives you would need to compute via autograd.
3. The IC for the wave equation requires both $u(0, x, y)$ and $u_t(0, x, y)$. How would you enforce the condition on $u_t$ in the PINN loss?
4. Compared to the 1D heat equation, how does the computational cost scale? Consider both the number of collocation points needed and the number of autograd calls per point.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """